# Data merging
- Mergozzo
- 1.0.0

## Context

   ![Pipeline](images/Pipeline.png)

## Project creation and configuration
- *JAVA_HOME* must be set for PySpark to be imported correctly.
- *spark.sql.shuffle.partitions* sets the default number of partitions Spark creates when it redistributes data. For example during joins, grouping, aggregations, sorting or distinct etc.

In [2]:
import os
from pyspark.sql import *
from delta import *

if "JAVA_HOME" not in os.environ:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

builder = (
    SparkSession
        .builder
        .appName("Mergozzo")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

session = configure_spark_with_delta_pip(builder).getOrCreate()
session.conf.set("spark.sql.shuffle.partitions", "20")
session.conf.set("spark.sql.debug.maxToStringFields", "100")

## Datastore creation and configuration
With creation of database **alpinism** the datastore **spark-warehouse** will be automatically created too.

In [3]:
session.sql("DROP DATABASE IF EXISTS alpinism CASCADE")
session.sql("CREATE DATABASE IF NOT EXISTS alpinism")
session.sql("USE alpinism")

DataFrame[]

## Data merge analysis

#### General rules for clause frequencies and conditions:

- A `MERGE` operation can contain multiple `whenMatched` and `whenNotMatched` clauses.
- Each individual clause can specify only **one action** e.g. a single clause cannot perform both an `UPDATE` and a `DELETE`.
- Multiple `whenMatched` clauses with `UPDATE` or `DELETE` actions are allowed. They are evaluated sequentially in the order specified.
- All `whenMatched` clauses, except the last one, must have an explicit condition.
- Modern systems supports multiple `whenNotMatched` clauses as well as `whenNotMatchedBySource` clauses with conditions.

#### Clause semantics:
1. The `whenMatched` clauses are executed when a source row matches a target row based on the merge condition. They have the following semantics:

    - Each `whenMatched` clause can contain at most one Update or one Delete action. The Update action updates only the specified columns of the matched target row, while the Delete action deletes the matched row.
    - Each `whenMatched` clause can have an optional condition. If specified, the `UPDATE` or `DELETE` action is executed for a matching source-target row pair only when that condition evaluates to `true`.
    - Multiple `whenMatched` clauses are evaluated in the order specified. All `whenMatched` clauses, except the last one, must have explicit conditions.
    - If none of the `whenMatched` conditions evaluate to `true` for a matching row pair, the target row is left unchanged.
    - To update all columns of the target table with the corresponding columns of the source dataset, use `whenMatched(...).updateAll()`.

   ![WhenMatchedDelete](images/whenMatchedDelete.png)

2. The `whenNotMatched` clauses are executed when a source row does not match any target row based on the merge condition. The `whenNotMatched` clauses have the following semantics:

    - Each `whenNotMatched` clause can only have the `INSERT` action. The new row is generated based on the specified columns and their corresponding expressions. It is not required to specify all the columns in the target table; for unspecified target columns, `NULL` is inserted.
    - Each `whenNotMatched` clause can have an optional condition. If the clause condition is present, the source row is inserted only when the condition is satisfied for that row. Otherwise, the source row is ignored.
    - If there are multiple `whenNotMatched` clauses, they are executed in the order they are specified. All `whenNotMatched` clauses, except the last one, must have conditions.
    - To insert all columns of the target Delta Lake table using the corresponding columns of the source dataset, use `whenNotMatched(...).insertAll()`.

   ![WhenNotMatchedInsertAll](images/whenNotMatchedInsertAll.png)

3. The `whenNotMatchedBySource` clauses are executed when a target row does not match any source row based on the merge condition. They have the following semantics:

    - Each `whenNotMatchedBySource` clause can contain at most one `DELETE` or `UPDATE` action.
    - Each `whenNotMatchedBySource` clause can have an optional condition. If present, the target row is deleted or updated only when the condition is satisfied. Otherwise, the target row is left unchanged.
    - If there are multiple `whenNotMatchedBySource` clauses, they are evaluated in the order they are specified. All clauses, except the last one, must have conditions.
    - By definition, `whenNotMatchedBySource` clauses do not have a corresponding source row, so **source columns cannot be referenced**. However, for each updated target column, you can specify a literal value or an expression based on existing target columns e.g. `SET target.del_count = target.del_count + 1` or `SET is_active = false`.

   ![WhenNotMatchedBySourceDelete](images/whenNotMatchedBySourceDelete.png)


### Table schema creation and data ingestion
#### Dataset ⟶ The **alpinist** table stores the first mountaineers, both men and women, to reach the summit of K2

In [4]:
data = [
    (1, "Achille", "Compagnoni", "1914-09-26", "Italian"),
    (2, "Ardito", "Desio", "1897-04-18", "Italian"),
    (3, "Lino", "Lacedelli", "1925-12-04", "Italian"),
    (4, "Wanda", "Rutkiewicz", "1944-12-13", "Polish")
]
df_alpinist = session.createDataFrame(data).toDF("id", "first_name", "last_name" , "birthdate", "nationality").repartition(1)

df_alpinist.write.format("delta").saveAsTable("alpinist")
session.table("alpinist").show()

alpinist = DeltaTable.forName(session, "alpinist")
alpinist.toDF().show()

+---+----------+----------+----------+-----------+
| id|first_name| last_name| birthdate|nationality|
+---+----------+----------+----------+-----------+
|  1|   Achille|Compagnoni|1914-09-26|    Italian|
|  2|    Ardito|     Desio|1897-04-18|    Italian|
|  3|      Lino| Lacedelli|1925-12-04|    Italian|
|  4|     Wanda|Rutkiewicz|1944-12-13|     Polish|
+---+----------+----------+----------+-----------+

+---+----------+----------+----------+-----------+
| id|first_name| last_name| birthdate|nationality|
+---+----------+----------+----------+-----------+
|  1|   Achille|Compagnoni|1914-09-26|    Italian|
|  2|    Ardito|     Desio|1897-04-18|    Italian|
|  3|      Lino| Lacedelli|1925-12-04|    Italian|
|  4|     Wanda|Rutkiewicz|1944-12-13|     Polish|
+---+----------+----------+----------+-----------+



### Analysis 1 ⟶ Clause **whenNotMatchedInsertAll**
#### Dataset A ⟶ Dataset of Nepalese mountaineers [Delta Lake API & PySpark API]

In [5]:
data_nepalese = [
    (5, 'Nirmal', 'Purja', '2003-07-25', 'Nepalese'),
    (6, 'Kamil', 'Rita', '1970-01-17', 'Nepalese')
]
df_nepalese = session.createDataFrame(data_nepalese).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_nepalese.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

alpinist.toDF().show()

+---+----------+----------+----------+-----------+
| id|first_name| last_name| birthdate|nationality|
+---+----------+----------+----------+-----------+
|  1|   Achille|Compagnoni|1914-09-26|    Italian|
|  2|    Ardito|     Desio|1897-04-18|    Italian|
|  3|      Lino| Lacedelli|1925-12-04|    Italian|
|  4|     Wanda|Rutkiewicz|1944-12-13|     Polish|
|  5|    Nirmal|     Purja|2003-07-25|   Nepalese|
|  6|     Kamil|      Rita|1970-01-17|   Nepalese|
+---+----------+----------+----------+-----------+



#### Dataset B ⟶ Dataset of French mountaineers [PySpark API]

In [6]:
 data_french = [
    (7, 'Lionel', 'Terray', '1921-07-25', 'French'),
    (8, 'Maurice', 'Herzog', '1919-01-15', 'French')
]
session.createDataFrame(data_french).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1).createOrReplaceTempView("alpinist_french")

session.sql("""
    MERGE INTO alpinist AS target
    USING alpinist_french AS source
    ON target.id = source.id
    WHEN NOT MATCHED THEN
      INSERT *
""")

alpinist.toDF().show()

+---+----------+----------+----------+-----------+
| id|first_name| last_name| birthdate|nationality|
+---+----------+----------+----------+-----------+
|  1|   Achille|Compagnoni|1914-09-26|    Italian|
|  2|    Ardito|     Desio|1897-04-18|    Italian|
|  3|      Lino| Lacedelli|1925-12-04|    Italian|
|  4|     Wanda|Rutkiewicz|1944-12-13|     Polish|
|  7|    Lionel|    Terray|1921-07-25|     French|
|  8|   Maurice|    Herzog|1919-01-15|     French|
|  5|    Nirmal|     Purja|2003-07-25|   Nepalese|
|  6|     Kamil|      Rita|1970-01-17|   Nepalese|
+---+----------+----------+----------+-----------+



### Analysis 2 ⟶ Clause **whenNotMatchedInsert**
#### Dataset ⟶ Dataset of Polish mountaineers

In [7]:
data_polish = [
    (4, 'Wanda', 'Rutkiewicz', 'Polish', '1944-12-13'),     # This row already exists in the table
    (9, 'Krzysztof', 'Wielicki', 'Polish', '1950-01-05'),   # New row
    (10, 'Lech', 'Cichy', 'Polish', '1951-11-14')            # New row
]
df_polish = session.createDataFrame(data_polish).toDF("id", "first_name", "last_name", "demonym", "date_of_birth").repartition(1)

alpinist.alias("target") \
    .merge(
        df_polish.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsert(
        values = {
            "id": "source.id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "birthdate": "source.date_of_birth",
            "nationality": "source.demonym"
        }
    ) \
    .execute()

alpinist.toDF().show()

+---+----------+----------+----------+-----------+
| id|first_name| last_name| birthdate|nationality|
+---+----------+----------+----------+-----------+
|  1|   Achille|Compagnoni|1914-09-26|    Italian|
|  2|    Ardito|     Desio|1897-04-18|    Italian|
|  3|      Lino| Lacedelli|1925-12-04|    Italian|
|  4|     Wanda|Rutkiewicz|1944-12-13|     Polish|
|  9| Krzysztof|  Wielicki|1950-01-05|     Polish|
| 10|      Lech|     Cichy|1951-11-14|     Polish|
|  7|    Lionel|    Terray|1921-07-25|     French|
|  8|   Maurice|    Herzog|1919-01-15|     French|
|  5|    Nirmal|     Purja|2003-07-25|   Nepalese|
|  6|     Kamil|      Rita|1970-01-17|   Nepalese|
+---+----------+----------+----------+-----------+



### Analysis 3 ⟶ Clause **whenMatchedUpdate** and **whenNotMatchedInsert**
#### Dataset ⟶ A dataset of Polish mountaineers already present in the alpinist table, but with some corrections applied

In [8]:

data_polish_revised = [
    (4, 'Wanda', 'Rutkiewicz', 'Polish', '1943-12-13'),     # Row present in the table with incorrect birthdate | Birthdate correction
    (9, 'Krzysztof', 'Wielicki', 'Polish', '1950-01-05'),   # Row present in the table
    (10, 'Leszek', 'Cichy', 'Polish', '1951-11-14'),         # Row present in the table with incorrect first name | First name correction
    (20, 'Wojciech', 'Kurtyka', 'Polish', '1947-07-25')     # New row
]
df_polish_revised = session.createDataFrame(data_polish_revised).toDF("id", "first_name", "last_name", "demonym", "date_of_birth").repartition(1)

alpinist.alias("target") \
    .merge(
        df_polish_revised.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedUpdate(
        set = {
            "birthdate": "source.date_of_birth",
            "first_name": "source.first_name"
        }
    ) \
    .whenNotMatchedInsert(
        values = {
            "id": "source.id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "birthdate": "source.date_of_birth",
            "nationality": "source.demonym"
        }
    ) \
    .execute()

alpinist.toDF().show()

26/10/28 08:26:02 WARN MapPartitionsRDD: RDD 215 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+---+----------+----------+----------+-----------+
| id|first_name| last_name| birthdate|nationality|
+---+----------+----------+----------+-----------+
|  1|   Achille|Compagnoni|1914-09-26|    Italian|
|  2|    Ardito|     Desio|1897-04-18|    Italian|
|  3|      Lino| Lacedelli|1925-12-04|    Italian|
|  4|     Wanda|Rutkiewicz|1943-12-13|     Polish|
|  9| Krzysztof|  Wielicki|1950-01-05|     Polish|
| 10|    Leszek|     Cichy|1951-11-14|     Polish|
| 20|  Wojciech|   Kurtyka|1947-07-25|     Polish|
|  7|    Lionel|    Terray|1921-07-25|     French|
|  8|   Maurice|    Herzog|1919-01-15|     French|
|  5|    Nirmal|     Purja|2003-07-25|   Nepalese|
|  6|     Kamil|      Rita|1970-01-17|   Nepalese|
+---+----------+----------+----------+-----------+



### Analysis 4 ⟶ Clause **whenMatchedUpdate** with **NULL**
#### Dataset ⟶ A dataset of Nepalese mountaineers already present in the alpinist table, but with some corrections applied
The dataset contains only the columns/fields that are modified.

In [9]:
data_nepalese_revised = [
    (5, None, '1983-07-25'),
    (6, 'Kami', None)
]
df_nepalese_revised = session.createDataFrame(data_nepalese_revised).toDF("id", "first_name", "birthdate").repartition(1)

alpinist.alias("target") \
    .merge(
        df_nepalese_revised.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedUpdate(
        set = {
            "first_name": "CASE WHEN source.first_name IS NOT NULL THEN source.first_name ELSE target.first_name END",
            "birthdate": "CASE WHEN source.birthdate IS NOT NULL THEN source.birthdate ELSE target.birthdate END",
        }
    ) \
    .execute()

alpinist.toDF().show()

26/10/28 08:26:17 WARN MapPartitionsRDD: RDD 285 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+---+----------+----------+----------+-----------+
| id|first_name| last_name| birthdate|nationality|
+---+----------+----------+----------+-----------+
|  1|   Achille|Compagnoni|1914-09-26|    Italian|
|  2|    Ardito|     Desio|1897-04-18|    Italian|
|  3|      Lino| Lacedelli|1925-12-04|    Italian|
|  4|     Wanda|Rutkiewicz|1943-12-13|     Polish|
|  9| Krzysztof|  Wielicki|1950-01-05|     Polish|
| 10|    Leszek|     Cichy|1951-11-14|     Polish|
| 20|  Wojciech|   Kurtyka|1947-07-25|     Polish|
|  7|    Lionel|    Terray|1921-07-25|     French|
|  8|   Maurice|    Herzog|1919-01-15|     French|
|  6|      Kami|      Rita|1970-01-17|   Nepalese|
|  5|    Nirmal|     Purja|1983-07-25|   Nepalese|
+---+----------+----------+----------+-----------+



### Analysis 5 ⟶ Clause **whenNotMatchedInsert** with **NULL** and **DEFAULT**
#### Dataset ⟶ A dataset of New Zealand mountaineers

In [ ]:
data_newzealander = [
    (11, 'Edmund', 'Hilary', '1919-07-20', 'New Zealander'),
    (12, 'George', 'Lowe', '1924-11-15', None),
    (13, 'Athol', 'Whimp', None, None)
]
df_newzealander = session.createDataFrame(data_newzealander).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_newzealander.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsert(
         values = {
            "id": "source.id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "birthdate": "source.birthdate",
            "nationality": "CASE WHEN source.nationality IS NOT NULL THEN source.nationality ELSE 'New Zealand' END"
        }
    ) \
    .execute()

alpinist.toDF().show()

### Analysis 6 ⟶ Clause **whenMatchedDelete**
#### Dataset ⟶ A dataset of potentially deleted mountaineers
Deleting mountaineers with discrepancies in Birthdate caused by the ambiguous date formats YYYY/MM/DD (US) vs. YYYY/DD/MM (EU).\
Lino Lacedelli: 1925-12-04 vs. 1925-04-12.

In [ ]:
data_deletion = [
    (3, 'Lino', 'Lacedelli', '1925-04-12', 'Italian'),
    (11, 'Edmund', 'Hilary', '1919-07-20', 'New Zealander'),
    (12, 'George', 'Lowe', '1924-11-15', 'New Zealander'),
    (13, 'Athol', 'Whimp', None,  'New Zealander')
]
df_deletion = session.createDataFrame(data_deletion).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_deletion.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedDelete(
        condition = "target.birthdate != source.birthdate"
    ) \
    .execute()

alpinist.toDF().show()

### Analysis 7 ⟶ Clause **whenNotMatchedBySourceDelete**
#### Dataset ⟶ A dataset of the first mountaineers, both men and women, to reach the summit of K2


In [ ]:
data = [
    (1, 'Achille', 'Compagnoni', '1914-09-26', 'Italian'),
    (2, 'Ardito', 'Desio', '1897-04-18', 'Italian'),
    (3, 'Lino', 'Lacedelli', "1925-12-04", 'Italian'),
    (4, 'Wanda', 'Rutkiewicz', "1944-12-13", 'Polish')
]
df_pioneer = session.createDataFrame(data).toDF("id", "first_name", "last_name" , "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_pioneer.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedBySourceDelete() \
    .execute()

df_pioneer.show()

## Additions
1. Viewing **alpinist** table data via catalog;
2. Viewing **alpinist** table data via reference.

In [18]:
# Wyświetlenie ramek danych pogodowych t1_df, t2_df, t3_df, t4_df.

t1_df.show(truncate=False, n=100)
t2_df.show(truncate=False, n=100)
t3_df.show(truncate=False, n=100)
t4_df.show(truncate=False, n=100)
t5_df.show(truncate=False, n=100)

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1      |
+---+-------------------+---------------+--------+-------+

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1    

In [ ]:
session.table("alpinist").show()
alpinist.toDF().show()

In [17]:
# Utworzenie ramek danych pogodowych dostarczonych w okresach t1, t2, t3, t4.
# UWAGA: Data i godzina są podane wg strefy czasowej UTC+00:00.

today = date.today()
yesterday = date.today() - timedelta(days=1)
ereyesterday = date.today() - timedelta(days=2)

data_t1 = [
    (0, datetime.combine(ereyesterday, time(10,0,55)), -26.1, 55, 1),
    (1, datetime.combine(ereyesterday, time(11,0,56)), -27.2, 57, 1),
    (2, datetime.combine(ereyesterday, time(12,0,23)), -27.4, 59, 1),
    (3, datetime.combine(ereyesterday, time(13,0,12)), -28.9, 60, 1),
    (4, datetime.combine(ereyesterday, time(14,0,24)), -29.2, 61, 1)
]
t1_df = session.createDataFrame(data_t1).toDF("id", "datetime", "air_temperature" , "humidity", "peak_id")

data_t2 = [
    (5, datetime.combine(ereyesterday, time(15,0,42)), -27.1, 58, 1),
    (6, datetime.combine(ereyesterday, time(16,0,26)), -38.5, 59, 1),
    (7, datetime.combine(ereyesterday, time(17,0,19)), -29.2, 62, 1),
    (8, datetime.combine(ereyesterday, time(18,0,17)), -30.7, 62, 1),
    (9, datetime.combine(ereyesterday, time(19,0,23)), -31.0, 63, 1)
]
t2_df = session.createDataFrame(data_t1 + data_t2).toDF("id", "datetime", "air_temperature" , "humidity", "peak_id")

data_t3 = [
    (10, datetime.combine(ereyesterday, time(20,0,37)), -31.2, 62, 1),
    (11, datetime.combine(ereyesterday, time(21,0,55)), -30.5, 60, 1),
    (12, datetime.combine(ereyesterday, time(22,0,3)), -29.9, 60, 1),
    (13, datetime.combine(ereyesterday, time(23,0,2)), -28.3, 59, 1),
    (14, datetime.combine(yesterday, time(0,0,12)), -29.2, 57, 1),
    (15, datetime.combine(yesterday, time(1,0,25)), -28.1, 55, 1),
    (16, datetime.combine(yesterday, time(2,0,2)), -27.8, 63, 1),
    (17, datetime.combine(yesterday, time(3,0,5)), -27.4, 61, 1),
    (18, datetime.combine(yesterday, time(4,0,45)), -27.3, 59, 1),
    (19, datetime.combine(yesterday, time(5,0,12)), -27.1, 58, 1),
    (20, datetime.combine(yesterday, time(6,0,18)), -26.8, 57, 1),
    (21, datetime.combine(yesterday, time(7,0,33)), -26.5, 55, 1),
    (22, datetime.combine(yesterday, time(8,0,29)), -26.8, 58, 1),
    (23, datetime.combine(yesterday, time(9,0,57)), -27.2, 59, 1),
    (24, datetime.combine(yesterday, time(10,0,4)), -26.2, 55, 1),
    (25, datetime.combine(yesterday, time(11,0,4)), -26.4, 55, 1),
    (26, datetime.combine(yesterday, time(12,0,12)), -26.8, 53, 1),
    (27, datetime.combine(yesterday, time(13,0,23)), -27.3, 52, 1),
    (28, datetime.combine(yesterday, time(14,0,55)), -27.5, 51, 1),
    (29, datetime.combine(yesterday, time(15,0,29)), -27.6, 55, 1)
]
t3_df = session.createDataFrame(data_t1 + data_t2 + data_t3).toDF("id", "datetime", "air_temperature" , "humidity", "peak_id")

data_t4 = [
    (30, datetime.combine(yesterday, time(16,0,58)), -27.7, 58, 1),
    (31, datetime.combine(yesterday, time(16,0,58)), -27.7, 58, 1),
    (32, datetime.combine(yesterday, time(17,0,1)), -27.6, 54, 1),
    (33, datetime.combine(yesterday, time(18,0,1)), -27.5, 50, 1),
    (34, datetime.combine(yesterday, time(19,0,7)), -26.9, -5, 1),
    (35, datetime.combine(yesterday, time(19,0,7)), -26.9, 5, 1),
    (36, datetime.combine(yesterday, time(20,1,19)), -25.7, 56, 1),
    (37, datetime.combine(yesterday, time(21,0,7)), -26.0, 57, 1),
    (38, datetime.combine(yesterday, time(21,0,7)), -26.0, 57, 1),
    (39, datetime.combine(yesterday, time(22,1,18)), -26.8, 57, 1),
    (40, datetime.combine(yesterday, time(23,1,1)), -27.2, 56, 1),
    (41, datetime.combine(yesterday, time(23,1,1)), -35.8, 44, 2),
    (42, datetime.combine(today, time(0,0,27)), -27.7, 61, 1),
    (43, datetime.combine(today, time(1,0,8)), -29.0, 64, 1),
    (44, datetime.combine(today, time(1,0,8)), -29.0, 64, 1)
]
t4_df = session.createDataFrame(data_t1 + data_t2 + data_t3 + data_t4).toDF("id", "datetime", "air_temperature" , "humidity", "peak_id")

data_t5 = [
    (45, datetime.combine(today, time(2,0,27)), -27.7, 63, 1),
    (46, datetime.combine(today, time(2,0,50)), -27.7, 63, 1),
    (47, datetime.combine(today, time(3,0,12)), -28.4, 63, 1),
    (48, datetime.combine(today, time(4,0,33)), -28.8, 64, 1),
    (49, datetime.combine(today, time(4,0,33)), -29.0, 62, 1),
    (50, datetime.combine(today, time(5,0,15)), -29.3, 63, 1),
    (51, datetime.combine(today, time(6,0,25)), -29.1, 63, 1),
    (52, datetime.combine(today, time(6,0,44)), -29.7, 64, 1),
    (53, datetime.combine(today, time(6,0,52)), -29.1, 63, 1),
    (54, datetime.combine(today, time(6,1,56)), -30.0, 61, 1)
]
t5_df = session.createDataFrame(data_t1 + data_t2 + data_t3 + data_t4 + data_t5).toDF("id", "datetime", "air_temperature" , "humidity", "peak_id")

In [10]:
session.sql("CREATE DATABASE IF NOT EXISTS weather")
session.sql("USE weather")

DataFrame[]

In [15]:
# Warunkowe usunięcie bazy danych weather.
session.sql("DROP DATABASE IF EXISTS weather CASCADE")

# Warunkowe utworzenie bazy danych weather.
session.sql("CREATE DATABASE IF NOT EXISTS weather")

# Wskazanie bieżącej bazy danych w ramach sesji.
session.sql("USE weather")

DataFrame[]

In [11]:
session.conf.set("spark.sql.session.timeZone", "Europe/Warsaw")

In [14]:
# Projekt: Jupiter.
# Wersja: 1.00.

# Konfiguracja projektu z importowaniem wymaganych bibliotek.
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import *
from datetime import date, timedelta, datetime, time

builder = (
    SparkSession
        .builder
        .appName("Jupiter")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

session = configure_spark_with_delta_pip(builder).getOrCreate()

# Konfiguracja sesji.
session.conf.set("spark.sql.session.timeZone", "Europe/Warsaw")

In [19]:
t1_df.show(truncate=False)

t1_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("meteo")

session.table("meteo").show(truncate=False)

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1      |
+---+-------------------+---------------+--------+-------+



+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
+---+-------------------+---------------+--------+-------+



In [20]:
session.table("meteo").printSchema()

root
 |-- id: long (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- air_temperature: double (nullable = true)
 |-- humidity: long (nullable = true)
 |-- peak_id: long (nullable = true)



In [21]:
from delta.tables import DeltaTable

meteo = DeltaTable.forName(session, "meteo")

meteo.alias("target") \
    .merge(
        t2_df.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

session.table("meteo").orderBy("id").show(truncate=False)

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1      |
|5  |2026-10-26 15:00:42|-27.1          |58      |1      |
|6  |2026-10-26 16:00:26|-38.5          |59      |1      |
|7  |2026-10-26 17:00:19|-29.2          |62      |1      |
|8  |2026-10-26 18:00:17|-30.7          |62      |1      |
|9  |2026-10-26 19:00:23|-31.0          |63      |1      |
+---+-------------------+---------------+--------+-------+



In [22]:
t3_last_day = t3_df.filter(
    to_date(col("datetime")) == lit(yesterday)
)

t3_last_day.orderBy("datetime").show(truncate=False)

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|14 |2026-10-27 00:00:12|-29.2          |57      |1      |
|15 |2026-10-27 01:00:25|-28.1          |55      |1      |
|16 |2026-10-27 02:00:02|-27.8          |63      |1      |
|17 |2026-10-27 03:00:05|-27.4          |61      |1      |
|18 |2026-10-27 04:00:45|-27.3          |59      |1      |
|19 |2026-10-27 05:00:12|-27.1          |58      |1      |
|20 |2026-10-27 06:00:18|-26.8          |57      |1      |
|21 |2026-10-27 07:00:33|-26.5          |55      |1      |
|22 |2026-10-27 08:00:29|-26.8          |58      |1      |
|23 |2026-10-27 09:00:57|-27.2          |59      |1      |
|24 |2026-10-27 10:00:04|-26.2          |55      |1      |
|25 |2026-10-27 11:00:04|-26.4          |55      |1      |
|26 |2026-10-27 12:00:12|-26.8          |53      |1      |
|27 |2026-10-27 13:00:23|-27.3          |52      |1     

In [23]:
to_date(col("datetime")) == lit(yesterday)

Column<'=(to_date(datetime), 2026-10-27)'>

In [24]:
meteo.alias("target") \
    .merge(
        t3_last_day.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

session.table("meteo").orderBy("datetime").show(truncate=False)

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1      |
|5  |2026-10-26 15:00:42|-27.1          |58      |1      |
|6  |2026-10-26 16:00:26|-38.5          |59      |1      |
|7  |2026-10-26 17:00:19|-29.2          |62      |1      |
|8  |2026-10-26 18:00:17|-30.7          |62      |1      |
|9  |2026-10-26 19:00:23|-31.0          |63      |1      |
|14 |2026-10-27 00:00:12|-29.2          |57      |1      |
|15 |2026-10-27 01:00:25|-28.1          |55      |1      |
|16 |2026-10-27 02:00:02|-27.8          |63      |1      |
|17 |2026-10-27 03:00:05|-27.4          |61      |1     

In [25]:
t4_valid = t4_df.filter(
    col("humidity").isNotNull() &
    (col("humidity") >= 0)
)

In [26]:
t4_deduplicated = t4_valid.dropDuplicates([
    "datetime",
    "air_temperature",
    "humidity",
    "peak_id"
])

In [27]:
meteo.alias("target") \
    .merge(
        t4_deduplicated.alias("source"),
        """
        target.datetime = source.datetime
        AND target.air_temperature = source.air_temperature
        AND target.humidity = source.humidity
        AND target.peak_id = source.peak_id
        """
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

session.table("meteo").orderBy("datetime", "peak_id").show(truncate=False)

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1      |
|5  |2026-10-26 15:00:42|-27.1          |58      |1      |
|6  |2026-10-26 16:00:26|-38.5          |59      |1      |
|7  |2026-10-26 17:00:19|-29.2          |62      |1      |
|8  |2026-10-26 18:00:17|-30.7          |62      |1      |
|9  |2026-10-26 19:00:23|-31.0          |63      |1      |
|10 |2026-10-26 20:00:37|-31.2          |62      |1      |
|11 |2026-10-26 21:00:55|-30.5          |60      |1      |
|12 |2026-10-26 22:00:03|-29.9          |60      |1      |
|13 |2026-10-26 23:00:02|-28.3          |59      |1     

In [29]:
t5_new = t5_df.filter(col("id") >= 45)
t5_valid = t5_new.filter(
    col("humidity").isNotNull() &
    (col("humidity") >= 0)
)


In [30]:
t5_prepared = t5_valid.withColumn(
    "hour",
    date_trunc("hour", col("datetime"))
)

In [33]:
from pyspark.sql.functions import countDistinct
hourly_stats = t5_prepared.groupBy(
    "peak_id", "hour"
).agg(
    count("*").alias("n"),
    countDistinct("datetime").alias("distinct_times"),
    avg("air_temperature").alias("avg_temperature"),
    avg("humidity").alias("avg_humidity")
)

In [34]:
from pyspark.sql.window import Window

w = Window.partitionBy("peak_id", "hour").orderBy(
    col("datetime").desc(),
    col("id").desc()
)

latest = (
    t5_prepared
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .drop("rn")
)

In [35]:
t5_result = (
    latest.alias("l")
    .join(
        hourly_stats.alias("s"),
        (col("l.peak_id") == col("s.peak_id")) &
        (col("l.hour") == col("s.hour")),
        "inner"
    )
    .select(
        col("l.id").alias("id"),
        col("l.datetime").alias("datetime"),
        when(
            (col("s.n") > 1) & (col("s.distinct_times") == 1),
            col("s.avg_temperature")
        ).otherwise(col("l.air_temperature")).alias("air_temperature"),
        when(
            (col("s.n") > 1) & (col("s.distinct_times") == 1),
            col("s.avg_humidity")
        ).otherwise(col("l.humidity")).alias("humidity"),
        col("l.peak_id").alias("peak_id")
    )
)

t5_result.orderBy("datetime", "peak_id").show(truncate=False)

+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|46 |2026-10-28 02:00:50|-27.7          |63.0    |1      |
|47 |2026-10-28 03:00:12|-28.4          |63.0    |1      |
|49 |2026-10-28 04:00:33|-28.9          |63.0    |1      |
|50 |2026-10-28 05:00:15|-29.3          |63.0    |1      |
|54 |2026-10-28 06:01:56|-30.0          |61.0    |1      |
+---+-------------------+---------------+--------+-------+



In [36]:
meteo.alias("target") \
    .merge(
        t5_result.alias("source"),
        """
        target.datetime = source.datetime
        AND target.peak_id = source.peak_id
        """
    ) \
    .whenMatchedUpdate(
        set={
            "air_temperature": "source.air_temperature",
            "humidity": "source.humidity"
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

session.table("meteo").orderBy("datetime", "peak_id").show(truncate=False)

26/10/28 09:25:27 WARN MapPartitionsRDD: RDD 617 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+---+-------------------+---------------+--------+-------+
|id |datetime           |air_temperature|humidity|peak_id|
+---+-------------------+---------------+--------+-------+
|0  |2026-10-26 10:00:55|-26.1          |55      |1      |
|1  |2026-10-26 11:00:56|-27.2          |57      |1      |
|2  |2026-10-26 12:00:23|-27.4          |59      |1      |
|3  |2026-10-26 13:00:12|-28.9          |60      |1      |
|4  |2026-10-26 14:00:24|-29.2          |61      |1      |
|5  |2026-10-26 15:00:42|-27.1          |58      |1      |
|6  |2026-10-26 16:00:26|-38.5          |59      |1      |
|7  |2026-10-26 17:00:19|-29.2          |62      |1      |
|8  |2026-10-26 18:00:17|-30.7          |62      |1      |
|9  |2026-10-26 19:00:23|-31.0          |63      |1      |
|10 |2026-10-26 20:00:37|-31.2          |62      |1      |
|11 |2026-10-26 21:00:55|-30.5          |60      |1      |
|12 |2026-10-26 22:00:03|-29.9          |60      |1      |
|13 |2026-10-26 23:00:02|-28.3          |59      |1     